## 1. 데이터 전처리

runpod에서 a100 GPU 1개 sxm 대여해주시고 container disk는 50기가 바이트

In [1]:
%pip install --upgrade torch torchvision torchaudio
%pip install "transformers>=4.50.0" --force-reinstall --no-deps
%pip install tokenizers safetensors huggingface-hub --upgrade
%pip install datasets accelerate trl peft --upgrade


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may n

In [2]:
# ImportError: cannot import name 'TypeIs' from 'typing_extensions' (/usr/local/lib/python3.11/dist-packages/typing_extensions.py)
# 에러 발생 시에는 

In [3]:
!pip install typing_extensions==4.11.0


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

In [5]:
!wget https://github.com/llm-fine-tuning/LLaMA-Factory/raw/refs/heads/main/data/esg_binary_classification_train.json
!wget https://github.com/llm-fine-tuning/LLaMA-Factory/raw/refs/heads/main/data/esg_binary_classification_test.json

--2026-02-20 07:33:36--  https://github.com/llm-fine-tuning/LLaMA-Factory/raw/refs/heads/main/data/esg_binary_classification_train.json
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/llm-fine-tuning/LLaMA-Factory/refs/heads/main/data/esg_binary_classification_train.json [following]
--2026-02-20 07:33:36--  https://raw.githubusercontent.com/llm-fine-tuning/LLaMA-Factory/refs/heads/main/data/esg_binary_classification_train.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10738972 (10M) [application/octet-stream]
Saving to: ‘esg_binary_classification_train.json’

esg_binary_classifi 100%[=====

In [17]:
import json
import random

# 1. JSON 파일 로드
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    return dataset

# 2. 데이터 로드
train_dataset = load_dataset('esg_binary_classification_train.json')
test_dataset = load_dataset('esg_binary_classification_test.json')

print(f"학습 데이터 개수: {len(train_dataset)}")
print(f"테스트 데이터 개수: {len(test_dataset)}")

학습 데이터 개수: 3728
테스트 데이터 개수: 932


In [18]:
# 3. 데이터 구조 확인
print("\n=== 데이터 구조 확인 ===")
print("첫 번째 학습 데이터 샘플:")
print(json.dumps(train_dataset[0], ensure_ascii=False, indent=2))


=== 데이터 구조 확인 ===
첫 번째 학습 데이터 샘플:
{
  "instruction": "LG에너지솔루션, 배터리 관리 분야 새 얼굴 '비 어라운드' 만들었다 LG에너지솔루션의 배터리 관리 토털 설루션(BMTS) 브랜드 'B.around'(비.어라운드) 로고. LG에너지솔루션 제공  잇단 화재로 전기차 배터리에 불안과 공포(전기차 포비아)가 확산한 가운데 LG에너지솔루션(LG엔솔)이 배터리 관리 토털 설루션(BMTS) 브랜드를 만들고 사업 확장에 나선다. 전기차 배터리 생산을 넘어 소프트웨어 부문 강화로 수익원을 넓히려는 움직임으로 풀이된다.  LG엔솔은 25일 '항상 고객의 곁에'(Be around your side)란 뜻의 BMTS 브랜드 'B.around'(비.어라운드)를 공개했다. 어떤 상황에서든 배터리 상태를 실시간 모니터링하는 LG엔솔 BMTS의 기술 경쟁력을 상징한다는 설명이다.  비.어라운드 제품에는 기존 배터리 관리 시스템(BMS)에 클라우드와 인공지능(AI) 기술을 결합해 화재방지 등을 위한 안전 진단, 퇴화·수명 예측 등을 더 쉽게 할 계획이다. 배터리 불량 유형을 사전에 진단하고 퇴화 상태를 점검하는 소프트웨어를 통해 오랫동안 쓸 수 있게 할 방침이다.  이달훈 LG에너지솔루션 BMS개발센터장 상무는 \"오랜 기간 안전 진단 시스템을 위해 노력했고 이제는 배터리 관리 소프트웨어 사업으로 확장해 배터리의 건강한 사용을 위해 설루션을 제공하는 기업이 되려고 한다\"고 말했다.  김청환 기자 chk@hankookilbo.com",
  "input": "",
  "output": "근거: LG에너지솔루션이 배터리 관리 토털 솔루션(BMTS) 브랜드인 'B.around'를 출시한 것은 ESG와 관련이 있습니다. 사회 측면에서는 화재 방지와 같은 안전 진단을 포함한 배터리 관리 강화는 제품 책임 및 안전을 향상시키기 위한 노력으로 볼 수 있습니다. 이는 고객의 신뢰를 높이고 제품의 안전성을 확보하는 사회적 책임 경영과 관련이 깊습니다. 또한

In [19]:
print("\n=== 데이터 구조 확인 ===")
print("첫 번째 테스트 데이터 샘플:")
print(json.dumps(test_dataset[0], ensure_ascii=False, indent=2))


=== 데이터 구조 확인 ===
첫 번째 테스트 데이터 샘플:
{
  "instruction": "\"월남쌈 대신 '비비고' 만두 먹고 CGV서 영화 봐요\"…'베트남 젠지' 홀린 CJ CJ그룹은 지난 10일부터 9월 1일까지 베트남에서 K컬처 축제 'CJ K FESTA'를 연다.ⓒ 뉴스1/김진희 기자.  (호찌민=뉴스1) 김진희 기자 = \"CJ(001040) 로고가 화려하고 예쁩니다. CGV를 통해 CJ를 알게 됐는데 젊고 역동적이어서 '젠지' 세대에 적합한 이미지라는 생각이 듭니다.\" -화이빠오(24·남)  지난 15~16일(현지시간) 베트남 호찌민 일대에서 열린 'CJ K FESTA'를 찾은 베트남 현지인들은 극찬을 쏟아냈다.  CJ그룹은 베트남에 진출한 계열사를 총동원해 CJ K FESTA를 올해 처음으로 개최했다. 이번 축제는 △K푸드 Week(8월 10~16일) △K스포츠 Week(8월 17~23일) △K무비 Week(8월 24일~9월 1일) 등 3가지 테마로 구성됐다.  CJ는 호찌민 소재 유명 마트와 쇼핑몰 내 CGV 매장에서 푸드쇼를 운영하며 브랜드 홍보에 적극 나섰다.  CJ그룹은 지난 10일부터 9월 1일까지 베트남에서 K컬처 축제 'CJ K FESTA'를 연다.ⓒ 뉴스1/김진희 기자.  15일 오후 호찌민 소재 메가마켓에는 더운 날씨임에도 마트 외부 공간에 사람들로 북적였다. CJ 제품을 선보이는 푸드트럭을 구경하기 위해 찾은 이들이었다. 한국과 달리 베트남에서는 푸드트럭이 생소하다고 한다.  이날 CJ는 CJ제일제당 비비고의 해산물 만두, 호떡, 김스낵과 뚜레쥬르의 빵을 선보였다. CJ제일제당의 자회사인 베트남 현지 냉동식품기업 '까우제'(Cautre)의 제품도 만나볼 수 있었다.  호찌민 시민 화이빠오는 \"예전에 만두를 많이 먹어 봤지만 비비고 만두는 독특하고 맛있다\"며 \"특히 푸드트럭을 처음 봐서 신기한데, 이렇게 화려하게 시식·광고하는 건 처음 본다\"고 말했다.  CJ그룹은 지난 10일부터 9월 1일까지 베트남에서 K컬

In [20]:
# 4. OpenAI format으로 데이터 변환을 위한 함수
system_prompt = '''1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.
2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.
- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)
순환 경제 (Circular Economy)
- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)
- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights Protection), 내부 통제 및 감사 (Internal Controls & Audits), 리스크 관리 (Risk Management), 투명한 정보 공개 (Transparent Disclosure), 기업 윤리 및 부패 방지 (Corporate Ethics & Anti-Corruption), 경영진 보상 구조 (Executive Compensation Structure), 컴플라이언스 (Regulatory Compliance), 이해관계자 참여 (Stakeholder Engagement)
3. 입력이 주어지면 근거를 작성하고 ESG 관련 기사이면 => True 아니면 => False를 작성합니다.
4. 답변은 항상 '=> True' 또는 '=> False'로 끝나야 하며 그 뒤에 아무 것도 적지마세요.
5. 지역의 정책이나 장소에 대한 내용이거나, 개인의 사건과 같이 특정 기업에 대한 이야기가 아닌 경우에는 아니라고 판단하세요. 저는 상장 '기업'의 내용에 집중하고 있습니다.
6. 기업의 경영권 분쟁과 같은 경우에는 단순히 개인의 이야기라고 볼 수 없습니다. 이 경우에도 ESG라고 판단할 수 있습니다.
7. 개인적인 부고의 기사는 ESG와 관련이 없다고 판단하세요.'''

def format_data(sample):
    return {
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user", 
                "content": sample["instruction"]  # instruction을 user prompt로 사용
            },
            {
                "role": "assistant",
                "content": sample["output"]  # output을 assistant response로 사용
            }
        ]
    }

In [21]:
# 5. 전체 데이터셋을 OpenAI 포맷으로 변환
def convert_to_openai_format(dataset):
    formatted_data = []
    for sample in dataset:
        try:
            formatted_sample = format_data(sample)
            formatted_data.append(formatted_sample)
        except KeyError as e:
            print(f"키 오류 발생: {e}")
            print(f"샘플 데이터: {sample}")
            break
    return formatted_data

In [22]:
# 6. 데이터 변환 실행
print("\n=== OpenAI 포맷으로 변환 중... ===")
train_dataset = convert_to_openai_format(train_dataset)
print(f"변환된 학습 데이터 개수: {len(train_dataset)}")


=== OpenAI 포맷으로 변환 중... ===
변환된 학습 데이터 개수: 3728


In [23]:
# 임의의 345번 학습 데이터 출력
train_dataset[345]

{'messages': [{'role': 'system',
   'content': "1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.\n2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.\n- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)\n순환 경제 (Circular Economy)\n- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)\n- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shar

In [45]:
test_dataset = convert_to_openai_format(test_dataset)
print(f"변환된 테스트 데이터 개수: {len(test_dataset)}")

변환된 테스트 데이터 개수: 932


In [46]:
# 임의의 345번 데이터 출력
test_dataset[345]

{'messages': [{'role': 'system',
   'content': "1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.\n2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.\n- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)\n순환 경제 (Circular Economy)\n- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)\n- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shar

In [25]:
# 리스트 형태에서 다시 Dataset 객체로 변경
print(type(train_dataset))
print(type(test_dataset))
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
print(type(train_dataset))
print(type(test_dataset))

<class 'list'>
<class 'list'>
<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


In [26]:
train_dataset[0]

{'messages': [{'content': "1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.\n2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.\n- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)\n순환 경제 (Circular Economy)\n- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)\n- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights Protec

In [48]:
test_dataset[0]

{'messages': [{'content': "1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.\n2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.\n- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)\n순환 경제 (Circular Economy)\n- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)\n- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights Protec

## 2. 모델 로드 및 템플릿 적용

In [27]:
# 허깅페이스 모델 ID
model_id = "Qwen/Qwen3-8B" 

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [28]:
# 템플릿 적용
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)
print(text)

<|im_start|>system
1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.
2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.
- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)
순환 경제 (Circular Economy)
- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)
- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights Protection), 내부 통제 

## 3. LoRA와 SFTConfig 설정

In [29]:
peft_config = LoraConfig(
        lora_alpha=32,
        lora_dropout=0.1,
        r=8,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
)

- lora_alpha: LoRA(Low-Rank Adaptation)에서 사용하는 스케일링 계수를 설정합니다. LoRA의 가중치 업데이트가 모델에 미치는 영향을 조정하는 역할을 하며, 일반적으로 학습 안정성과 관련이 있습니다.
- lora_dropout: LoRA 적용 시 드롭아웃 확률을 설정합니다. 드롭아웃은 과적합(overfitting)을 방지하기 위해 일부 뉴런을 랜덤하게 비활성화하는 정규화 기법입니다. 0.1로 설정하면 학습 중 10%의 뉴런이 비활성화.
- r: LoRA의 랭크(rank)를 설정합니다. 이는 LoRA가 학습할 저차원 공간의 크기를 결정합니다. 작은 값일수록 계산 및 메모리 효율이 높아지지만 모델의 학습 능력이 제한될 수 있습니다.
- bias: LoRA 적용 시 편향(bias) 처리 방식을 지정합니다. "none"으로 설정하면 편향이 LoRA에 의해 조정되지 않습니다. "all" 또는 "lora_only"와 같은 값으로 변경하여 편향을 조정할 수도 있습니다.
- target_modules: LoRA를 적용할 특정 모듈(레이어)의 이름을 리스트로 지정합니다. 예제에서는 "q_proj"와 "v_proj"를 지정하여, 주로 Self-Attention 메커니즘의 쿼리와 값 프로젝션 부분에 LoRA를 적용합니다.
- task_type: LoRA가 적용되는 작업 유형을 지정합니다. "CAUSAL_LM"은 Causal Language Modeling, 즉 시퀀스 생성 작업에 해당합니다. 다른 예로는 "SEQ2SEQ_LM"(시퀀스-투-시퀀스 언어 모델링) 등이 있습니다.

In [30]:
# 최대 길이
max_seq_length=16384

In [41]:
args = SFTConfig(
    output_dir="qwen3-8b-esg-binary-model",           # 저장될 디렉토리와 저장소 ID
    num_train_epochs=2,                      # 학습할 총 에포크 수 
    per_device_train_batch_size=1,           # GPU당 배치 크기
    gradient_accumulation_steps=8,           # 그래디언트 누적 스텝 수
    gradient_checkpointing=True,             # 메모리 절약을 위한 체크포인팅
    optim="adamw_torch_fused",               # 최적화기
    logging_steps=10,                        # 로그 기록 주기
    save_strategy="steps",                   # 저장 전략
    save_steps=50,                           # 저장 주기
    bf16=True,                              # bfloat16 사용
    learning_rate=1e-4,                     # 학습률
    max_grad_norm=0.3,                      # 그래디언트 클리핑
    warmup_ratio=0.03,                      # 워밍업 비율
    lr_scheduler_type="constant",           # 고정 학습률
    push_to_hub=False,                      # 허브 업로드 안 함
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=[], # 이제는 report_to=None이 안 됨.
    max_length=max_seq_length,              # 최대 시퀀스 길이 추가
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


- `output_dir`: 학습 결과가 저장될 디렉토리 또는 모델 저장소의 이름을 지정합니다. 이 디렉토리에 학습된 모델 가중치, 설정 파일, 로그 파일 등이 저장됩니다.

- `num_train_epochs`: 모델을 학습시키는 총 에포크(epoch) 수를 지정합니다. 에포크는 학습 데이터 전체를 한 번 순회한 주기를 의미합니다. 예를 들어, `3`으로 설정하면 데이터셋을 3번 학습합니.

- `per_device_train_batch_size`: GPU 한 대당 사용되는 배치(batch)의 크기를 설정합니다. 배치 크기는 모델이 한 번에 처리하는 데이터 샘플의 수를 의미합니다. 작은 크기는 메모리 사용량이 적지만 학습 시간이 증가할 수 있니다.

- `gradient_accumulation_steps`: 그래디언트를 누적할 스텝(step) 수를 지정합니다. 이 값이 `2`로 설정된 경우, 두 스텝마다 그래디언트를 업데이트합니다. 배치 크기를 가상으로 늘리는 효과가 있으며, GPU 메모리 부족 문제를 해결할 때 용합니다.

- `gradient_checkpointing`: 그래디언트 체크포인팅을 활성화하여 메모리를 절약합니다. 이 옵션은 계산 그래프를 일부 저장하지 않고 다시 계산하여 메모리를 절약하지만, 속도가 약간 느려질수 있습니다.

- `optim`: 학습 시 사용할 최적화 알고리즘을 설정합니다. `adamw_torch_fused`는 PyTorch의 효율적인 AdamW 최적기를 사용합니다.

- `logging_steps`: 로그를 기록하는 주기를 스텝 단위로 지정합니다. 예를 들어, `10`으로 설정하면 매 10 스텝마 로그를 기록합니다.

- `save_strategy`: 모델을 저장하는 전략을 설정합니다. `"steps"`로 설정된 경우, 지정된 스마다 모델이 저장됩니다.

- `save_steps`: 모델을 저장하는 주기를 스텝 단위로 설정합니다. 예를 들어, `50`으로 설정하면 매 50스텝마다 모델을 저장합니다.

- `bf16`: `bfloat16` 정밀도를 사용하도록 설정합니다. `bfloat16`은 FP32와 유사한 범위를 제공하면서 모리와 계산 효율성을 높입니다.

- `learning_rate`: 학습률을 지정합니다. 학습률은 모델의 가중치가 한 번의 업데이트에서 얼마나 크게 변할지를 결정합니다. 일반적으로 작은 값을 용하여 안정적인 학습을 유도합니다.

- `max_grad_norm`: 그래디언트 클리핑의 임계값을 설정합니다. 이 값보다 큰 그래디언트가 발생하면, 임계값으로 정하여 폭발적 그래디언트를 방지합니다.

- `warmup_ratio`: 학습 초기 단계에서 학습률을 선형으로 증가시키는 워밍업 비율을 지정합니다 학습의 안정성을 높이기 위해 사용됩니다.

- `lr_scheduler_type`: 학습률 스케줄러의 유형을 설정합니다. `"costant"`는 학습률을 일정하게 유지합니다.

- `push_to_hub`: 학습된 모델을 허브에 업로드할지 여부를 설정합니. `False`로 설정하면 업로드하지 않습니다.

- `remove_unused_columns`: 사용되지 않는 열을 제거할지 여부를 설정합니다.`True`로 설정하면 메모리를 절약할 수 있습니다.

- `dataset_kwargs`: 데이터셋 로딩 시 추가적인 설정을 전달합니다. 예제에서는 `skip_prepare_dataset True`로 설정하여 데이터셋 준비 단계를 건너뜁니다.

- `report_to`: 학습 로그를 보고할 대상을 지정합니다. `None`으로 설정되면 로그가 기록되지 않습니다.

## 4. 학습 중 전처리 함수: collate_fn

In [32]:
def collate_fn(batch):
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }
    
    for example in batch:
        # messages의 각 내용에서 개행문자 제거
        clean_messages = []
        for message in example["messages"]:
            clean_message = {
                "role": message["role"],
                "content": message["content"]
            }
            clean_messages.append(clean_message)
        
        # 깨끗해진 메시지로 템플릿 적용
        text = tokenizer.apply_chat_template(
            clean_messages,
            tokenize=False,
            add_generation_prompt=False
        ).strip()
        
        # 텍스트를 토큰화
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )
        
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        
        # 레이블 초기화
        labels = [-100] * len(input_ids)
        
        # assistant 응답 부분 찾기
        im_start = "<|im_start|>"
        im_end = "<|im_end|>"
        assistant = "assistant"
        
        # 토큰 ID 가져오기
        im_start_tokens = tokenizer.encode(im_start, add_special_tokens=False)
        im_end_tokens = tokenizer.encode(im_end, add_special_tokens=False)
        assistant_tokens = tokenizer.encode(assistant, add_special_tokens=False)
        
        i = 0
        while i < len(input_ids):
            # <|im_start|>assistant 찾기
            if (i + len(im_start_tokens) <= len(input_ids) and 
                input_ids[i:i+len(im_start_tokens)] == im_start_tokens):
                
                # assistant 토큰 찾기
                assistant_pos = i + len(im_start_tokens)
                if (assistant_pos + len(assistant_tokens) <= len(input_ids) and 
                    input_ids[assistant_pos:assistant_pos+len(assistant_tokens)] == assistant_tokens):
                    
                    # assistant 응답의 시작 위치로 이동
                    current_pos = assistant_pos + len(assistant_tokens)
                    
                    # <|im_end|>를 찾을 때까지 레이블 설정
                    while current_pos < len(input_ids):
                        if (current_pos + len(im_end_tokens) <= len(input_ids) and 
                            input_ids[current_pos:current_pos+len(im_end_tokens)] == im_end_tokens):
                            # <|im_end|> 토큰도 레이블에 포함
                            for j in range(len(im_end_tokens)):
                                labels[current_pos + j] = input_ids[current_pos + j]
                            break
                        labels[current_pos] = input_ids[current_pos]
                        current_pos += 1
                    
                    i = current_pos
                
            i += 1
        
        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)
    
    # 패딩 적용
    max_length = max(len(ids) for ids in new_batch["input_ids"])
    
    for i in range(len(new_batch["input_ids"])):
        padding_length = max_length - len(new_batch["input_ids"][i])
        
        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * padding_length)
        new_batch["attention_mask"][i].extend([0] * padding_length)
        new_batch["labels"][i].extend([-100] * padding_length)
    
    # 텐서로 변환
    for k, v in new_batch.items():
        new_batch[k] = torch.tensor(v)
    
    return new_batch

collate_fn(batch) 함수는 자연어 처리 모델 학습을 위해 데이터를 전처리하는 역할을 수행합니다. 이 함수는 배치 내의 데이터를 처리하여 모델이 사용할 수 있는 입력 형식으로 변환합니다.

먼저, 각 샘플의 메시지에서 개행 문자를 제거하고 필요한 정보만 남깁니다. 정리된 메시지로 텍스트를 구성하고 이를 토큰화하여 input_ids와 attention_mask를 생성합니다. 이후 assistant 답변 부분을 찾아 해당 범위에 레이블을 설정합니다. 이 범위를 제외한 나머지 위치는 -100으로 설정하여 손실 계산에서 제외되도록 합니다.

최종적으로, 배치 내 모든 샘플의 길이를 동일하게 맞추기 위해 패딩 작업을 수행합니다. 이 과정에서 입력 데이터에는 패딩 토큰 ID를 추가하고, 어텐션 마스크에는 0을 추가하며, 레이블에는 -100을 추가합니다. 모든 데이터는 PyTorch 텐서로 변환되어 반환됩니다.

In [33]:
# collate_fn 테스트 (배치 크기 1로)
example = train_dataset[0]
batch = collate_fn([example])

print("\n처리된 배치 데이터:")
print("입력 ID 형태:", batch["input_ids"].shape)
print("어텐션 마스크 형태:", batch["attention_mask"].shape)
print("레이블 형태:", batch["labels"].shape)


처리된 배치 데이터:
입력 ID 형태: torch.Size([1, 1282])
어텐션 마스크 형태: torch.Size([1, 1282])
레이블 형태: torch.Size([1, 1282])


In [34]:
print('입력에 대한 정수 인코딩 결과:')
print(batch["input_ids"][0].tolist())

입력에 대한 정수 인코딩 결과:
[151644, 8948, 198, 16, 13, 58034, 40853, 55054, 20401, 5140, 231, 112, 24897, 18411, 63332, 34395, 468, 7783, 129985, 54116, 55054, 134039, 140569, 127268, 57268, 624, 17, 13, 468, 7783, 126804, 126317, 19391, 128605, 20136, 113, 125512, 10764, 92120, 130109, 29346, 16560, 136646, 20401, 220, 16, 15, 59761, 10764, 92120, 130109, 29346, 18411, 142616, 96137, 140569, 127268, 57268, 624, 12, 46832, 246, 65306, 320, 82066, 8, 549, 74361, 226, 43590, 73669, 69923, 129423, 43590, 320, 36607, 468, 2728, 58100, 701, 129242, 76435, 90486, 127085, 21329, 40720, 320, 34625, 365, 480, 12354, 24567, 701, 64577, 54321, 141540, 132841, 32831, 320, 4783, 66567, 701, 69441, 238, 20487, 126251, 92751, 28002, 320, 54, 5525, 9551, 701, 90711, 250, 132892, 138017, 73523, 126835, 320, 36, 1015, 7276, 17784, 5643, 10816, 701, 54116, 127033, 137525, 60960, 131518, 320, 82046, 10388, 21766, 17930, 701, 73077, 137284, 74808, 21329, 320, 49207, 1488, 35847, 701, 47818, 126251, 135391, 32831, 6

In [35]:
# 디코딩된 input_ids 출력
decoded_text = tokenizer.decode(
    batch["input_ids"][0].tolist(),
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False
)

print("\ninput_ids 디코딩 결과:")
print(decoded_text)


input_ids 디코딩 결과:
<|im_start|>system
1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.
2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.
- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)
순환 경제 (Circular Economy)
- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)
- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights 

In [36]:
print('레이블에 대한 정수 인코딩 결과:')
print(batch["labels"][0].tolist())

레이블에 대한 정수 인코딩 결과:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

In [37]:
# -100이 아닌 부분만 골라 디코딩
label_ids = [token_id for token_id in batch["labels"][0].tolist() if token_id != -100]

decoded_labels = tokenizer.decode(
    label_ids,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False
)

print("\nlabels 디코딩 결과 (-100 제외):")
print(decoded_labels)


labels 디코딩 결과 (-100 제외):

<think>

</think>

근거: LG에너지솔루션이 배터리 관리 토털 솔루션(BMTS) 브랜드인 'B.around'를 출시한 것은 ESG와 관련이 있습니다. 사회 측면에서는 화재 방지와 같은 안전 진단을 포함한 배터리 관리 강화는 제품 책임 및 안전을 향상시키기 위한 노력으로 볼 수 있습니다. 이는 고객의 신뢰를 높이고 제품의 안전성을 확보하는 사회적 책임 경영과 관련이 깊습니다. 또한, 기술적 혁신을 통해 전기차 배터리의 수명을 예측하고 퇴화 상태를 진단하는 것은 자원 효율성을 향상시키는 환경적 측면도 포함하고 있습니다. 
=> True<|im_end|>


### input_ids와 labels는 어떻게 생성되는가?

LLM 학습에서 `input_ids`와 `labels`는 모델의 학습 목표에 따라 생성됩니다. 이를 예시 문장과 정수 인코딩을 통해 상세히 설명하겠습니다.

예를 들어, 다음과 같은 대화 데이터를 모델이 학습해야 한다고 가정합니다.
사용자가 `안녕하세요, 오늘 날씨는 어떤가요?`라고 물었고,
모델은 `안녕하세요! 오늘 날씨는 맑고 화창합니다.`라고 응답해야 한다고 합시다.

LLaMA 3에서는 다음과 같은 템플릿 구조를 사용합니다:

`<|begin_of_text|><|start_header_id|>user<|end_header_id|>
안녕하세요, 오늘 날씨는 어떤가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|>
안녕하세요! 오늘 날씨는 맑고 화창합니다.<|eot_id|>`

이 전체 텍스트는 토크나이저에 의해 정수 시퀀스로 변환됩니다. 예시로 단순화된 정수 시퀀스는 다음과 같다고 가정합니다:

`input_ids = [1001, 2001, 3001, 4001, 5001, 6001, 7001, 1002, 1001, 8001, 9001, 1003, 2002]`

여기서 모델이 예측해야 할 영역은 assistant의 응답 부분인
`안녕하세요! 오늘 날씨는 맑고 화창합니다.`에 해당하는 토큰들입니다.
따라서 `labels`는 다음과 같이 설정됩니다:

`labels = [-100, -100, -100, -100, -100, -100, -100, -100, -100, 8001, 9001, 1003, 2002]`

이처럼 `labels`는 모델의 출력이 필요한 영역만을 포함하고, 나머지 부분은 `-100`으로 채워져
모델이 실제로 예측하고 오차를 계산해야 하는 대상(학습 대상)에서 제외됩니다.

이를 통해 모델은 불필요한 입력 부분을 학습하지 않고, assnt 응답 부분에만 집중할 수 있습니다.
"""


## 5. 학습

In [42]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [43]:
# 학습 시작
trainer.train()   # 모델이 자동으로 허브와 output_dir에 저장됨

# 모델 저장
trainer.save_model()   # 최종 모델을 저장

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.290369
20,0.909597
30,0.808156
40,0.756429
50,0.724790
60,0.723965
70,0.726317
80,0.699570
90,0.694289
100,0.658345


## 6. 테스트 데이터 준비하기

[{'content': "1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.\n2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.\n- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)\n순환 경제 (Circular Economy)\n- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)\n- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights Protection), 내부 통제 

In [63]:
prompt_lst = []
label_lst = []

assistant_tag = "<|im_start|>assistant\n"
eot_tag = "<|im_end|>"

for messages in test_dataset["messages"]:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt, rest = text.split(assistant_tag, 1)
    label = rest.split(eot_tag, 1)[0]

    prompt_lst.append(prompt + assistant_tag)
    label_lst.append(label)

In [64]:
print(prompt_lst[200])

<|im_start|>system
1. 상장사의 뉴스를 보고 ESG 관련 기사인지 판단하시오.
2. ESG 각각에 대한 핵심 키워드는 아래의 10개 키워드를 참고해서 판단하시오.
- 환경 (Environmental) : 탄소 배출 감소 (Carbon Emission Reduction), 재생 에너지 사용 (Renewable Energy Usage), 자원 효율성 (Resource Efficiency), 폐기물 관리 (Waste Management), 친환경 제품 개발 (Eco-Friendly Product Development), 기후 변화 대응 (Climate Change Mitigation), 오염 방지 (Pollution Prevention), 생물 다양성 보호 (Biodiversity Protection), 물 관리 (Water Management)
순환 경제 (Circular Economy)
- 사회 (Social) : 노동 인권 보호 (Labor Rights Protection), 다양성 및 포용성 (Diversity and Inclusion), 직원 안전 및 건강 (Employee Health & Safety), 공정한 노동 관행 (Fair Labor Practices), 커뮤니티 참여 및 개발 (Community Engagement & Development), 제품 책임 및 안전 (Product Responsibility & Safety), 공급망 책임 (Supply Chain Responsibility), 데이터 보호 및 프라이버시 (Data Protection & Privacy), 인재 개발 및 교육 (Talent Development & Education), 고객 만족도 (Customer Satisfaction)
- 지배구조 (Governance) : 이사회 독립성 (Board Independence), 윤리적 경영 (Ethical Management), 주주 권리 보호 (Shareholder Rights Protection), 내부 통제 

In [65]:
print(label_lst[200])

<think>

</think>

근거: 이 기사는 주로 기업의 재무 실적과 사업 전략에 관한 내용으로, ESG와 직접적으로 관련된 요소는 명확하게 드러나지 않습니다. 기사에 언급된 내용들은 환경, 사회, 지배구조 측면의 활동이나 영향보다는 주로 매출, 영업이익, 콘텐츠 전략, 시장 점유율 등에 집중되어 있습니다. 따라서 이 기사는 ESG 관련 기사라고 판단하기 어렵습니다. 
=> False


## 7. 파인튜닝 모델 테스트

`AutoPeftModelForCausalLM()`의 입력으로 `LoRA Adapter`가 저장된 체크포인트의 주소를 넣으면 `LoRA Adapter`가 기존의 LLM과 부착되어 로드됩니다. 이 과정은 `LoRA Adapter`의 가중치를 사전 학습된 언어 모델(LLM)에 통합하여 미세 조정된 모델을 완성하는 것을 의미합니다.

`peft_model_id` 변수는 미세 조정된 가중치가 저장된 체크포인트의 경로를 나타냅니다. `"qwen3-8b-esg-binary-model/checkpoint-932"`는 `LoRA Adapter` 가중치가 저장된 위치로, 이 경로에서 해당 가중치를 불러옵니다.

`fine_tuned_model`은 `AutoPeftModelForCausalLM.from_pretrained` 메서드를 통해 체크포인트를 로드하여 생성됩니다. 이 메서드는 LLM과 `LoRA Adapter`를 결합하고, 최적화된 설정으로 모델을 메모리에 로드합니다. `device_map="auto"` 옵션은 모델을 자동으로 GPU에 배치합니다.

`pipeline`은 Hugging Face의 고수준 유틸리티로, NLP 작업(예: 텍스트 생성, 번역, 요약 등)을 간단히 수행할 수 있게 해줍니다. 이 코드에서 사용된 `pipeline("text-generation")`은 텍스트 생성 작업을 수행하기 위한 파이프라인 객체를 생성합니다. 파이프라인은 내부적으로 모델과 토크나이저를 관리하여, 입력 텍스트를 토큰화하고, 모델을 통해 생성된 결과를 다시 디코딩하여 사람이 읽을 수 있는 텍스트로 변환합니다.

이 코드는 미세 조정된 LLM을 로드한 뒤, 이를 이용해 텍스트 생성 작업을 간단히 수행할 수 있도록 준비하는 데 목적이 있습니다. `pipeline`을 통해 텍스트 생성 작업을 실행하면, 입력 텍스트에 기반하여 모델이 다음 토큰을 예측하고 이를 반복적으로 생성합니다. 이 과정은 사용자에게 자연스러운 텍스트를 출력하는 데 사용됩니다.데 사용됩니다.

In [66]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import  AutoTokenizer, pipeline

In [67]:
peft_model_id = "qwen3-8b-esg-binary-model/checkpoint-932"
fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(peft_model_id, device_map="auto", torch_dtype=torch.float16)
pipe = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [68]:
eos_token = tokenizer("<|im_end|>",add_special_tokens=False)["input_ids"][0]

In [69]:
def test_inference(pipe, prompt):
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    return outputs[0]['generated_text'][len(prompt):].strip()

In [70]:
for prompt, label in zip(prompt_lst[10:15], label_lst[10:15]):
    # print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(pipe, prompt)}")
    print(f"    label:\n{label}")
    print("-"*50)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'eos_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
<think>

</think>

근거: 이 기사는 넥슨의 게임 서비스 종료 및 후속작 운영 계획에 관한 내용으로, ESG와 직접적인 관련이 없습니다. 환경, 사회, 지배구조 측면에서의 ESG 핵심 키워드와 관련된 내용이 포함되어 있지 않습니다. 
=> False
    label:
<think>

</think>

근거: 해당 기사는 넥슨의 온라인 게임 '카트라이더' 서비스 종료와 후속작 '카트라이더 드리프트'에 대한 내용입니다. 이는 주로 게임 서비스와 관련된 사업 전략에 관한 이야기로, ESG(환경, 사회, 지배구조)와 직접적인 관련이 없습니다. 따라서 ESG 관련 기사로 보기 어렵습니다. 
=> False
--------------------------------------------------


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
<think>

</think>

근거: 이 기사는 태영그룹의 에코비트 매각과 태영건설의 경영 정상화에 관한 내용으로, ESG와 관련이 있습니다. 첫째, 환경 측면에서 에코비트는 태영그룹의 핵심 자산으로, 에코비트의 매각은 태영건설의 재무 개선과 경영 정상화에 기여할 수 있습니다. 이는 기업의 지속 가능한 경영과 관련이 있습니다. 둘째, 지배구조 측면에서 태영건설의 워크아웃 이행과 자구계획은 기업의 리스크 관리와 내부 통제 및 감사와 관련이 있으며, 이는 ESG의 지배구조 원칙에 부합합니다. 따라서 이 기사는 ESG와 관련이 있습니다. 
=> True
    label:
<think>

</think>

근거: 티와이홀딩스의 에코비트 매각은 ESG와 관련이 있습니다. 지배구조 측면에서 보면, 이는 기업의 리스크 관리와 내부 통제 및 구조조정을 위한 활동의 일환입니다. 태영건설의 경영 정상화와 재무지표 개선은 기업 지배구조의 투명성과 책임성을 높이는 조치로 볼 수 있으며, 이는 기업의 지속 가능성을 향상시키는 데 기여할 수 있습니다. 경영 정상화를 위한 자산 매각도 지배구조 개선의 일환으로 해석될 수 있습니다. 
=> True
--------------------------------------------------


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
<think>

</think>

근거: 한화호텔앤드리조트의 고객 예약 정보 유출 사건은 ESG 중 사회(Social) 측면과 관련이 있습니다. 고객의 개인정보 유출은 데이터 보호 및 프라이버시(Data Protection & Privacy)와 관련된 문제로, 이는 사회적 책임과 직결됩니다. 기업이 고객의 개인정보를 보호하지 못한 것은 사회적 신뢰를 저해하는 행위로, 이에 대한 사과와 재발 방지 대책이 필요합니다. 이러한 사고는 기업의 사회적 책임 경영과 관련이 있습니다. 
=> True
    label:
<think>

</think>

근거: 한화호텔앤드리조트가 시스템 오류로 고객의 예약정보 등 개인정보를 노출시킨 것은 ESG의 사회적 측면과 관련이 있습니다. 특히, 데이터 보호 및 프라이버시(Data Protection & Privacy)의 측면에서 중요합니다. 기업이 개인정보 유출 사고에 대해 사과하고, 재발 방지 대책을 마련하며 피해 고객에게 보상안을 제시하겠다고 밝히는 것은 데이터 보호 책임을 다하는 것으로 볼 수 있습니다. 이런 노력은 ESG의 사회적 책임을 실천하는 중요한 요소입니다. 
=> True
--------------------------------------------------


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
<think>

</think>

근거: 대한항공과 아시아나항공의 합병과 관련한 EU의 2단계 심사가 진행되는 내용은 ESG와 관련이 있습니다. 이는 지배구조 측면에서 기업의 경영권과 관련된 중요한 이슈로, 기업의 윤리적 경영과 투명한 정보 공개, 이해관계자 참여 등과 관련이 있습니다. 또한, 시장 경쟁 제한 우려로 인한 심사 진행은 리스크 관리와 관련된 사항으로 볼 수 있습니다. 
=> True
    label:
<think>

</think>

근거: 대한항공과 아시아나항공의 합병에 관한 내용은 기업 간의 인수 및 합병 과정에서 발생하는 시장 경쟁 제한 우려와 관련된 것으로, 이는 지배구조 측면의 리스크 관리 및 이해관계자 참여와 관련이 있습니다. 특히, 유럽연합(EU)의 기업결합 심층 심사는 기업의 지배구조와 관련된 규제와 컴플라이언스 측면에서 중요한 요소입니다. 
=> True
--------------------------------------------------
    response:
<think>

</think>

근거: 삼성전자의 감산 선언은 ESG와 관련이 있습니다. 환경 측면에서는 반도체 생산량을 줄이는 것은 자원 효율성을 높이는 행위로 볼 수 있습니다. 사회적 측면에서는 감산으로 인해 협력업체와 직원들의 어려움이 커질 수 있다는 우려가 제기되고 있어, 이는 사회적 책임 경영과 관련이 있습니다. 또한, 지배구조 측면에서는 기업의 전략적 결정이 투명하게 공개되고 있어 투명한 정보 공개와 관련이 있습니다. 
=> True
    label:
<think>

</think>

근거: 해당 기사는 삼성전자의 경영 전략과 관련된 내용이 있으며, 이는 ESG의 지배구조 측면과 관련이 있을 수 있습니다. 반도체 시장의 불확실성 속에서 감산을 결정한 것은 리스크 관리 및 주주 권리 보호와 관련된 경영 전략이라고 볼 수 있습니다. 그러나, 기사에서 주로 언급되는 것은 시장 상황과 주가 변동에 관한 것이며, 주요한 ESG 요소를